### Modèle YOLO
À partir de l'arboresence créée dans constitutiondataset.ipynb

In [ ]:
# 0. IMPORTATION DES LIBRAIRIES

#! pip install ultralytics
from pathlib import Path
import random
import shutil
import torch
from ultralytics import YOLO
import yaml

#reproductibilité
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
#On donne l'accès au drive pour récupérer les images
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. CONFIGURATION

DATA_YAML = Path("/content/drive/MyDrive/YOLO/datacolab.yaml") #chemin d'accès fichier YAML
MODEL_NAME = "yolo11n.pt" #Version choisie de YOLO

EPOCHS = 200 # Nombre d'itérations pour l'entraînement
IMAGE_SIZE = 1280 # Taille des images d'entrée
BATCH_SIZE = 16 # Taille du batch pour l'entraînement
SEED = 42 # Graine pour la reproducibilité

PROJECT_DIR = Path("runs/grazpedwri")
RUN_NAME = "detection_avec_boites_200_epochs_1280"
RUN_DIR = Path("runs/grazpedwri") / RUN_NAME

if not DATA_YAML.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {DATA_YAML.resolve()}"
    )

with open(DATA_YAML, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f) # Chargement de la configuration depuis le fichier YAML

print("Configuration du dataset :")
print(yaml.dump(config, allow_unicode=True))

print("GPU disponible :", torch.cuda.is_available()) # recherche de GPU disponible




In [ ]:
# 2. ENTRAINEMENT AVEC BOITES ET SAUVEGARDE TOUTES LES 5 EPOCHS

model = YOLO(MODEL_NAME) # Version du modèle a utiliser pour l'entrainement

results = model.train(
    data=str(DATA_YAML.resolve()), # chemin vers le fichier YAML de données
    epochs=EPOCHS, # Nombre d'itérations pour l'entraînement
    imgsz=IMAGE_SIZE, # Taille des images d'entrée
    batch=BATCH_SIZE, # Taille du batch pour l'entraînement
    seed=SEED, # Graine pour la reproducibilité
    deterministic=True, # pour la reproductibilité des résultats
    pretrained=True, # utiliser le modèle pré-entrainé sur COCO (jeu de données d'images ultralytics)
    patience=50, # nombre d'itérations sans amélioration avant arrêt
    project=str(PROJECT_DIR),# nom du projet pour l'enregistrement des résultats
    name=RUN_NAME, # nom du run pour l'enregistrement des résultats
    exist_ok=False, # si le dossier existe déjà, ne pas écraser les résultats précédents
    plots=True,
    save=True,
    save_period=5, # sauvegarde automatique toutes ls 5 epochs
    device=0 if torch.cuda.is_available() else "cpu",
)

In [ ]:
# 3. ÉVALUATION SUR LE TEST

best_model_path = (
    RUN_DIR / "weights" / "best.pt"
)

if not best_model_path.exists():
    raise FileNotFoundError(
        f"Poids du meilleur modèle introuvables : {best_model_path}"
    )

best_model = YOLO(str(best_model_path)) # Chargement du meilleur modèle pour l'évaluation

metrics = best_model.val(
    data=str(DATA_YAML.resolve()),
    split="test",
    imgsz=IMAGE_SIZE,
    plots=True,
    project=str(PROJECT_DIR),
    name="evaluation_test",
) # Evaluation du modèle sur le jeu de test

print("\nRésultats de détection :")
print("Précision :", metrics.box.mp)
print("Rappel    :", metrics.box.mr)
print("mAP@50    :", metrics.box.map50)
print("mAP@50:95 :", metrics.box.map)


### Sauvegarde du Run vers Google Drive

In [ ]:
source_dir = Path("/content") / PROJECT_DIR # Dossier source contenant les résultats à sauvegarder
destination_dir = Path("/content/drive/MyDrive") / PROJECT_DIR # Dossier de destination sur Google Drive

if source_dir.exists():
    destination_dir.parent.mkdir(parents=True, exist_ok=True)

    shutil.copytree(
        source_dir,
        destination_dir,
        dirs_exist_ok=True
    )

    print(f"Résultats sauvegardés dans : {destination_dir}")
else:
    print("Erreur : dossier de résultats introuvable.")